In [19]:
import pandas as pd
import requests

sample_df = pd.read_csv("sample_for_parsing.csv", dtype={"ИНН": str})
test_inn = sample_df["ИНН"].iloc[0]
print("Тестовый ИНН:", test_inn)

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
}

session.headers.update(headers)

try:
    response = session.get(
        f"https://bo.nalog.ru/advanced-search/organizations/search?query={test_inn}&page=0",
        timeout=15
    )
    
    print("Код ответа:", response.status_code)
    
    if response.status_code == 200:
        print(" Успешно!")
        print(response.text[:1000])
    else:
        print(" Ошибка:", response.status_code)
        print("Текст:", response.text[:500])
        
except Exception as e:
    print("Ошибка:", e)

Тестовый ИНН: 0268049176
Код ответа: 200
 Успешно!
<!doctype html><html xmlns="http://www.w3.org/1999/xhtml" lang="ru"><head><meta charset="utf-8"><meta content="" name="Description"><meta content="" name="Keywords"><meta http-equiv="X-UA-Compatible" content="IE=edge"><meta name="viewport" content="width=device-width,initial-scale=1"><link rel="shortcut icon" type="image/jpg" href="favicon.ico"/><title>Ð ÐµÑÑÑÑ ÐÐ¤Ð</title><link href="/static/css/main.93141287.css" rel="stylesheet"></head><body><div id="root" class="root"></div><div id="modal"></div><div id="side"></div><script type="text/javascript" src="/static/js/main.e815beb0.js"></script></body></html>


In [20]:
import requests
import json

url = "https://bo.nalog.gov.ru/advanced-search/organizations/search?query=0268049176&page=0&size=20"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://bo.nalog.gov.ru/",
}

session = requests.Session()
session.headers.update(headers)

response = session.get(url)

print("Код ответа:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("\n Успешно!")
    print("\nКлючи в ответе:", data.keys())
    print("\nСодержимое (первые 500 символов):")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:1000])
else:
    print("Ошибка:", response.text)

Код ответа: 200

 Успешно!

Ключи в ответе: dict_keys(['content', 'pageable', 'totalPages', 'totalElements', 'last', 'numberOfElements', 'first', 'size', 'number', 'sort', 'empty'])

Содержимое (первые 500 символов):
{
  "content": [
    {
      "id": 4866936,
      "inn": "<strong>0268049176</strong>",
      "shortName": "ООО \"БАШСПЕЦСТРОЙРЕМОНТ\"",
      "ogrn": "1080268002814",
      "index": "453130",
      "region": "БАШКОРТОСТАН",
      "district": null,
      "city": "СТЕРЛИТАМАК",
      "settlement": null,
      "street": "САГИТОВА",
      "house": "2Д",
      "building": null,
      "office": "3",
      "okved2": "41.2",
      "okopf": 12300,
      "okato": null,
      "okpo": null,
      "okfs": null,
      "statusCode": "ACTIVE",
      "statusDate": "2008-09-09",
      "bfo": {
        "period": "2025",
        "actualBfoDate": "2026-03-27",
        "gainSum": 41576,
        "knd": "0710099",
        "hasAz": false,
        "hasKs": false,
        "actualCorrectionNumber": 

In [14]:
import requests
import pandas as pd
import time
import json

def get_company_data(inn):
    """Получает данные о компании по ИНН"""
    
    url = f"https://bo.nalog.gov.ru/advanced-search/organizations/search?query={inn}&page=0&size=20"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://bo.nalog.gov.ru/",
    }
    
    result = {
        'id': None,
        'inn': inn,
        'name': 'Неизвестно',
        'ogrn': '',
        'region': '',
        'city': '',
        'okved2': '',
        'status': '',
        'period': '',
        'revenue': 0,
        'error': None
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            
            if data['content']:
                company = data['content'][0]
                bfo = company.get('bfo', {})
                
                revenue = bfo.get('gainSum')
                if revenue is None:
                    revenue = 0
                
                result.update({
                    'id': company.get('id'),
                    'name': company.get('shortName', 'Неизвестно'),
                    'ogrn': company.get('ogrn', ''),
                    'region': company.get('region', ''),
                    'city': company.get('city', ''),
                    'okved2': company.get('okved2', ''),
                    'status': company.get('statusCode', ''),
                    'period': bfo.get('period', ''),
                    'revenue': revenue,
                    'error': None
                })
            else:
                result['name'] = 'Не найдена'
                result['error'] = 'Компания не найдена'
        else:
            result['name'] = 'Ошибка HTTP'
            result['error'] = f'HTTP {response.status_code}'
            
    except Exception as e:
        result['name'] = 'Ошибка запроса'
        result['error'] = str(e)
    
    return result

In [15]:
test_result = get_company_data("0268049176")
print(test_result)

{'id': 4866936, 'inn': '0268049176', 'name': 'ООО "БАШСПЕЦСТРОЙРЕМОНТ"', 'ogrn': '1080268002814', 'region': 'БАШКОРТОСТАН', 'city': 'СТЕРЛИТАМАК', 'okved2': '41.2', 'status': 'ACTIVE', 'period': '2025', 'revenue': 41576, 'error': None}


In [16]:
sample_df = pd.read_csv('sample_for_parsing.csv', dtype={'ИНН': str})
print(f"Всего компаний: {len(sample_df)}")

Всего компаний: 5000


In [17]:
test_inns = sample_df['ИНН'].tolist()[:5]
print(f"Тест на {len(test_inns)} компаниях")

results = []
for i, inn in enumerate(test_inns):
    print(f"Обработка {i+1}/{len(test_inns)}: {inn}")
    result = get_company_data(inn)
    results.append(result)
    
    if result['error']:
        print(f"  Ошибка: {result['error']}")
    elif result['revenue'] > 0:
        print(f"  {result['name']} - Выручка: {result['revenue']} тыс. руб")
    else:
        print(f"  Нет данных о выручке")
    
    time.sleep(0.5)

test_df = pd.DataFrame(results)
test_df.to_csv('test_parsed.csv', index=False)

print("Тест завершен")
print(test_df[['inn', 'name', 'revenue', 'error']])

Тест на 5 компаниях
Обработка 1/5: 0268049176
  ООО "БАШСПЕЦСТРОЙРЕМОНТ" - Выручка: 41576 тыс. руб
Обработка 2/5: 6612030370
  Нет данных о выручке
Обработка 3/5: 5404041266
  ООО "ФОРТУНА +" - Выручка: 684670 тыс. руб
Обработка 4/5: 1649016260
  ООО "СТРОЙМОНТАЖ-СЕРВИС" - Выручка: 302525 тыс. руб
Обработка 5/5: 3234051052
  Нет данных о выручке
Тест завершен
          inn                      name  revenue error
0  0268049176  ООО "БАШСПЕЦСТРОЙРЕМОНТ"    41576  None
1  6612030370           ООО "СК "ОНИКА"        0  None
2  5404041266           ООО "ФОРТУНА +"   684670  None
3  1649016260  ООО "СТРОЙМОНТАЖ-СЕРВИС"   302525  None
4  3234051052  ООО "БРЯНСКСТРОЙПОДРЯД+"        0  None


In [18]:
all_inns = sample_df['ИНН'].tolist()
total = len(all_inns)

print(f"Запускаем полный парсинг {total} компаний")

results = []
start_time = time.time()

for i, inn in enumerate(all_inns):
    percent = (i + 1) / total * 100
    print(f"[{i+1}/{total}] ({percent:.1f}%) ИНН: {inn}")
    
    result = get_company_data(inn)
    results.append(result)
    
    if result['error']:
        print(f"  Ошибка: {result['error']}")
    elif result['revenue'] > 0:
        print(f"  {result['name']} - {result['revenue']} тыс. руб")
    else:
        print(f"  Нет данных о выручке")
    
    if (i + 1) % 50 == 0:
        pd.DataFrame(results).to_csv('parsed_progress.csv', index=False)
        elapsed = time.time() - start_time
        print(f"  Сохранен прогресс: {i+1}/{total}")
        print(f"  Времени прошло: {elapsed/60:.1f} минут")
    
    time.sleep(0.5)

final_df = pd.DataFrame(results)
final_df.to_csv('parsed_companies_full.csv', index=False)

elapsed_total = time.time() - start_time
print("Парсинг завершен")
print(f"Всего обработано: {len(results)} компаний")
print(f"С выручкой: {final_df[final_df['revenue'] > 0].shape[0]}")
print(f"С ошибками: {final_df[final_df['error'].notna()].shape[0]}")
print(f"Время: {elapsed_total/60:.1f} минут")

Запускаем полный парсинг 5000 компаний
[1/5000] (0.0%) ИНН: 0268049176
  ООО "БАШСПЕЦСТРОЙРЕМОНТ" - 41576 тыс. руб
[2/5000] (0.0%) ИНН: 6612030370
  Нет данных о выручке
[3/5000] (0.1%) ИНН: 5404041266
  ООО "ФОРТУНА +" - 684670 тыс. руб
[4/5000] (0.1%) ИНН: 1649016260
  ООО "СТРОЙМОНТАЖ-СЕРВИС" - 302525 тыс. руб
[5/5000] (0.1%) ИНН: 3234051052
  Нет данных о выручке
[6/5000] (0.1%) ИНН: 6730033514
  ООО СЗ "ГРАЖДАНСТРОЙ" - 114657 тыс. руб
[7/5000] (0.1%) ИНН: 7809006739
  ЗАО "ТРЕСТ СЗКС" - 71450 тыс. руб
[8/5000] (0.2%) ИНН: 7811421411
  ООО "СК "ГИДРОКОР" - 2443642 тыс. руб
[9/5000] (0.2%) ИНН: 7612044649
  ООО "АКАС" - 36382 тыс. руб
[10/5000] (0.2%) ИНН: 6230083651
  ООО "ВЕРТИКАЛЬ" - 29910 тыс. руб
[11/5000] (0.2%) ИНН: 7816671990
  ООО "РОСТЭНЕРГО" - 5854 тыс. руб
[12/5000] (0.2%) ИНН: 0238006023
  ООО "СТРОЙВЫСОТА" - 12315 тыс. руб
[13/5000] (0.3%) ИНН: 2016079306
  ООО "ИМПЕРИЯ" - 1040844 тыс. руб
[14/5000] (0.3%) ИНН: 7707460559
  ООО "ИНВЕСТПРОЕКТ" - 227395 тыс. руб
[15/5000

In [ ]:
def extract_year_metrics(year_entry):
    """Достаёт нужные показатели из одного годового отчёта."""
    period = year_entry.get("period")

    corrections = year_entry.get("typeCorrections", [])
    if not corrections:
        return None 

    fr = corrections[0].get("correction", {}).get("financialResult")
    if not fr:
        return None

    revenue = fr.get("current2110")             
    profit_before_tax = fr.get("current2300")  
    interest_payable = fr.get("current2330", 0.0) or 0.0  
    net_profit = fr.get("current2400")          

    if profit_before_tax is not None:
        ebit = profit_before_tax + interest_payable
        ebit_type = "full"
    elif net_profit is not None:
        ebit = net_profit
        ebit_type = "simplified"
    else:
        return None

    return {
        "period": period,
        "revenue": revenue,
        "net_profit": net_profit,
        "ebit": ebit,
        "ebit_type": ebit_type,
    }

In [20]:
def get_financials(org_id):
    """Получает финансовые показатели компании по всем доступным годам."""
    url = f"https://bo.nalog.gov.ru/nbo/organizations/{org_id}/bfo/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": "https://bo.nalog.gov.ru/",
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            return {"org_id": org_id, "error": f"HTTP {response.status_code}", "years": []}

        data = response.json()
        years = []
        for year_entry in data:
            metrics = extract_year_metrics(year_entry)
            if metrics is not None:
                years.append(metrics)

        return {"org_id": org_id, "error": None, "years": years}

    except Exception as e:
        return {"org_id": org_id, "error": str(e), "years": []}

In [22]:
test_result = get_financials(4866936)
print(test_result)

{'org_id': 4866936, 'error': None, 'years': [{'period': '2025', 'revenue': 41576.0, 'net_profit': 4660.0, 'ebit': 5008.0, 'ebit_type': 'full'}, {'period': '2024', 'revenue': 33825.0, 'net_profit': -12547.0, 'ebit': -12363.0, 'ebit_type': 'full'}, {'period': '2023', 'revenue': 245784.0, 'net_profit': 3237.0, 'ebit': 3761.0, 'ebit_type': 'full'}, {'period': '2022', 'revenue': 203017.0, 'net_profit': 1373.0, 'ebit': 1754.0, 'ebit_type': 'full'}, {'period': '2021', 'revenue': 57531.0, 'net_profit': 307.0, 'ebit': 497.0, 'ebit_type': 'full'}]}


In [11]:
import time

parsed_ids = pd.read_csv("parsed_companies_full.csv")
print(parsed_ids.shape)
parsed_ids.head()

(5000, 11)


,id,inn,name,ogrn,region,city,okved2,status,period,revenue,error
0,4866936.0,268049176,"ООО ""БАШСПЕЦСТРОЙРЕМОНТ""",1.080268e+12,БАШКОРТОСТАН,СТЕРЛИТАМАК,41.2,ACTIVE,2025.0,41576,NaN
1,4489050.0,6612030370,"ООО ""СК ""ОНИКА""",1.096612e+12,СВЕРДЛОВСКАЯ,КАМЕНСК-УРАЛЬСКИЙ,41.20,ACTIVE,2025.0,0,NaN
2,9874526.0,5404041266,"ООО ""ФОРТУНА +""",1.165476e+12,НОВОСИБИРСКАЯ,НОВОСИБИРСК,42.21,ACTIVE,2025.0,684670,NaN
3,5596177.0,1649016260,"ООО ""СТРОЙМОНТАЖ-СЕРВИС""",1.081689e+12,ТАТАРСТАН,NaN,49.41,ACTIVE,2025.0,302525,NaN
4,776486.0,3234051052,"ООО ""БРЯНСКСТРОЙПОДРЯД+""",1.033265e+12,МОСКВА,NaN,41.20,LIQUIDATION_STAGE,2025.0,0,NaN


In [12]:
test_ids = parsed_ids[parsed_ids["error"].isna()]["id"].head(5).tolist()
print("Тестовые id:", test_ids)

test_results = []
for i, org_id in enumerate(test_ids):
    print(f"{i+1}/{len(test_ids)}: org_id={org_id}")
    result = get_financials(int(org_id))
    test_results.append(result)
    if result["error"]:
        print("  Ошибка:", result["error"])
    else:
        print(f"  Лет с данными: {len(result['years'])}")
    time.sleep(0.5)

test_results

Тестовые id: [4866936.0, 4489050.0, 9874526.0, 5596177.0, 776486.0]
1/5: org_id=4866936.0
  Лет с данными: 5
2/5: org_id=4489050.0
  Лет с данными: 5
3/5: org_id=9874526.0
  Лет с данными: 5
4/5: org_id=5596177.0
  Лет с данными: 5
5/5: org_id=776486.0
  Лет с данными: 5


[{'org_id': 4866936,
  'error': None,
  'years': [{'period': '2025',
    'revenue': 41576.0,
    'net_profit': 4660.0,
    'ebit': 5008.0,
    'ebit_type': 'full'},
   {'period': '2024',
    'revenue': 33825.0,
    'net_profit': -12547.0,
    'ebit': -12363.0,
    'ebit_type': 'full'},
   {'period': '2023',
    'revenue': 245784.0,
    'net_profit': 3237.0,
    'ebit': 3761.0,
    'ebit_type': 'full'},
   {'period': '2022',
    'revenue': 203017.0,
    'net_profit': 1373.0,
    'ebit': 1754.0,
    'ebit_type': 'full'},
   {'period': '2021',
    'revenue': 57531.0,
    'net_profit': 307.0,
    'ebit': 497.0,
    'ebit_type': 'full'}]},
 {'org_id': 4489050,
  'error': None,
  'years': [{'period': '2025',
    'revenue': 0.0,
    'net_profit': -131.0,
    'ebit': -131.0,
    'ebit_type': 'full'},
   {'period': '2024',
    'revenue': 0.0,
    'net_profit': -131.0,
    'ebit': -131.0,
    'ebit_type': 'full'},
   {'period': '2023',
    'revenue': 0.0,
    'net_profit': -131.0,
    'ebit': -1

In [13]:
import json

all_ids = parsed_ids[parsed_ids["error"].isna()]["id"].tolist()
total = len(all_ids)
print(f"Компаний для парсинга финансов: {total}")

all_financials = []
start_time = time.time()

for i, org_id in enumerate(all_ids):
    result = get_financials(int(org_id))
    all_financials.append(result)

    percent = (i + 1) / total * 100
    if result["error"]:
        print(f"[{i+1}/{total}] ({percent:.1f}%) org_id={org_id} — ОШИБКА: {result['error']}")
    else:
        print(f"[{i+1}/{total}] ({percent:.1f}%) org_id={org_id} — {len(result['years'])} лет данных")

    if (i + 1) % 100 == 0:
        with open("financials_progress.json", "w", encoding="utf-8") as f:
            json.dump(all_financials, f, ensure_ascii=False)
        elapsed = time.time() - start_time
        print(f"  --- Прогресс сохранён: {i+1}/{total}, прошло {elapsed/60:.1f} мин ---")

    time.sleep(0.5)

with open("financials_full.json", "w", encoding="utf-8") as f:
    json.dump(all_financials, f, ensure_ascii=False)

elapsed_total = time.time() - start_time
print(f"\nГотово! Обработано {len(all_financials)} компаний за {elapsed_total/60:.1f} минут")

Компаний для парсинга финансов: 4957
[1/4957] (0.0%) org_id=4866936.0 — 5 лет данных
[2/4957] (0.0%) org_id=4489050.0 — 5 лет данных
[3/4957] (0.1%) org_id=9874526.0 — 5 лет данных
[4/4957] (0.1%) org_id=5596177.0 — 5 лет данных
[5/4957] (0.1%) org_id=776486.0 — 5 лет данных
[6/4957] (0.1%) org_id=3157427.0 — 5 лет данных
[7/4957] (0.1%) org_id=3958143.0 — 5 лет данных
[8/4957] (0.2%) org_id=5397109.0 — 5 лет данных
[9/4957] (0.2%) org_id=1103050.0 — 5 лет данных
[10/4957] (0.2%) org_id=8428748.0 — 5 лет данных
[11/4957] (0.2%) org_id=10653963.0 — 5 лет данных
[12/4957] (0.2%) org_id=10084588.0 — 5 лет данных
[13/4957] (0.3%) org_id=6364053.0 — 5 лет данных
[14/4957] (0.3%) org_id=11740597.0 — 2 лет данных
[15/4957] (0.3%) org_id=11533333.0 — 5 лет данных
[16/4957] (0.3%) org_id=9307746.0 — 5 лет данных
[17/4957] (0.3%) org_id=3690582.0 — 5 лет данных
[18/4957] (0.4%) org_id=8872268.0 — 5 лет данных
[19/4957] (0.4%) org_id=883128.0 — 5 лет данных
[20/4957] (0.4%) org_id=11531988.0 — 5 

In [14]:
import json

with open("financials_full.json", encoding="utf-8") as f:
    all_financials = json.load(f)

errors_count = sum(1 for c in all_financials if c["error"])
print("Компаний с ошибками при запросе:", errors_count)

years_counts = [len(c["years"]) for c in all_financials if not c["error"]]
import pandas as pd
print("\nРаспределение количества лет данных на компанию:")
print(pd.Series(years_counts).value_counts().sort_index())

full_count = sum(
    1 for c in all_financials if not c["error"]
    for y in c["years"] if y["ebit_type"] == "full"
)
simplified_count = sum(
    1 for c in all_financials if not c["error"]
    for y in c["years"] if y["ebit_type"] == "simplified"
)
print(f"\nЗаписей (компания-год) с полной отчётностью: {full_count}")
print(f"Записей (компания-год) с упрощённой отчётностью: {simplified_count}")

Компаний с ошибками при запросе: 142

Распределение количества лет данных на компанию:
0      26
1      92
2     257
3     318
4     421
5    3701
Name: count, dtype: int64

Записей (компания-год) с полной отчётностью: 13069
Записей (компания-год) с упрощённой отчётностью: 8680


In [15]:
period_sets = [
    frozenset(y["period"] for y in c["years"])
    for c in all_financials if not c["error"] and c["years"]
]

from collections import Counter
period_combo_counts = Counter(period_sets)

print("Самые частые наборы годов:")
for combo, count in period_combo_counts.most_common(10):
    print(sorted(combo), "—", count, "компаний")

Самые частые наборы годов:
['2021', '2022', '2023', '2024', '2025'] — 3701 компаний
['2021', '2022', '2023', '2024'] — 272 компаний
['2021', '2022', '2023'] — 267 компаний
['2021', '2022'] — 204 компаний
['2022', '2023', '2024', '2025'] — 86 компаний
['2022', '2023'] — 44 компаний
['2021'] — 43 компаний
['2022'] — 39 компаний
['2021', '2022', '2023', '2025'] — 29 компаний
['2022', '2023', '2024'] — 21 компаний


In [16]:
example = next(c for c in all_financials if not c["error"] and len(c["years"]) == 5)
print("org_id:", example["org_id"])
for year in example["years"]:
    print(year)

org_id: 4866936
{'period': '2025', 'revenue': 41576.0, 'net_profit': 4660.0, 'ebit': 5008.0, 'ebit_type': 'full'}
{'period': '2024', 'revenue': 33825.0, 'net_profit': -12547.0, 'ebit': -12363.0, 'ebit_type': 'full'}
{'period': '2023', 'revenue': 245784.0, 'net_profit': 3237.0, 'ebit': 3761.0, 'ebit_type': 'full'}
{'period': '2022', 'revenue': 203017.0, 'net_profit': 1373.0, 'ebit': 1754.0, 'ebit_type': 'full'}
{'period': '2021', 'revenue': 57531.0, 'net_profit': 307.0, 'ebit': 497.0, 'ebit_type': 'full'}
